# kubescan — full reproduce pipeline (Colab GPU)

Runs the same pipeline as `make reproduce` (see repo `Makefile`), unmodified — this notebook only
clones the repo, installs dependencies, and shells out to the existing scripts in `research/`.
No training code lives in this notebook; it stays the single source of truth in the repo.

**Before running:** `Runtime > Change runtime type > GPU` (A100/V100/T4 depending on your Colab tier).

In [ ]:
BRANCH = "feat/first-commit"  # change if the pipeline has since merged to main

!git clone --branch $BRANCH https://github.com/ObedRav/kubescan.git
%cd kubescan

In [ ]:
# torch itself is left alone — Colab already ships a CUDA-matched build.
# torch-geometric has no compiled-extension dependency in this codebase
# (no torch_scatter/torch_sparse/torch_cluster imports), so a plain pip
# install against Colab's existing torch is sufficient.
!pip install -q torch-geometric scikit-learn skops networkx

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > GPU"
print("CUDA device:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

In [ ]:
# Optional but recommended: Colab runtimes are ephemeral, so mount Drive
# to persist checkpoints/results after the session ends.
from google.colab import drive

drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/kubescan_checkpoints"
!mkdir -p "$DRIVE_OUT"

## Regenerate augmented graph data

`research/data/graphs/*_aug_*.npz` and `graphs_cache.npz` are gitignored (large, regenerable —
see `.gitignore`). The 97 original cluster graphs and `graph_manifest.csv` **are** tracked, so
augmentation is fully reproducible with a fixed seed. Splits (`research/data/splits/*.txt`) are
also tracked and must **not** be regenerated here — reusing them keeps folds identical to local runs.

In [ ]:
!python research/scripts/03_augment/augment_graphs.py --seed 42
!python research/scripts/04_build_datasets/build_graph_cache.py

## Train — RF → GNN (5-fold CV, GPU) → GA ensemble → test evaluation

Mirrors `make reproduce` minus the `data` step (done above) and the redundant fixed-split GNN
pass (removed from the Makefile — `gnn_fold_*.pt` from the CV loop is all downstream steps use).

In [ ]:
!python research/models/train_rf.py --seed 42

In [ ]:
# The GPU-bound step. resolve_device() auto-detects CUDA, and
# dataloader_kwargs() enables num_workers/pin_memory only on CUDA
# (on MPS/CPU workers regress performance on in-memory PyG datasets).
%cd research/models
!python train_gnn.py --cv-folds 5 --epochs 300 --hidden 64 --heads 4 --layers 3 --seed 42
%cd /content/kubescan

In [ ]:
%cd research/models
!python run_ga_ensemble.py --oof --seed 42
%cd /content/kubescan

In [ ]:
%cd research/models
!python evaluate_test_set.py --show-rankings
%cd /content/kubescan

In [ ]:
!python research/scripts/snapshot_run_manifest.py

## Persist results

Copies checkpoints + results JSON to the mounted Drive folder, and offers a zip download as a
fallback if Drive wasn't mounted.

In [ ]:
import os

if os.path.isdir("/content/drive/MyDrive"):
    !cp -r research/models/checkpoints/* "$DRIVE_OUT/"
    print(f"Copied checkpoints to {DRIVE_OUT}")
else:
    !zip -r kubescan_checkpoints.zip research/models/checkpoints
    from google.colab import files
    files.download("kubescan_checkpoints.zip")